In [24]:
import json
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from augur.utils import json_to_tree

## Separate all HA type trees into lineages
Use the all HA (like all H1 for instance) tree to identify lineages by eye, and then use the node that is common ancestor of that lineage to identify all tips that are a member of the lienage and write out a data file to make a lineage-specific tree

In [4]:
lineage_ancestral_node = {'H1': ['NODE_0000288', 'NODE_0000268'], 
                          'H3': ['NODE_0000889','NODE_0000499','NODE_0000483']}

In [17]:
# give each lineage within the HA type a letter
lineage_letters = {'H1': {'NODE_0000288': 'a', 'NODE_0000268': 'b'}, 
                          'H3': {'NODE_0000889': 'a', 'NODE_0000499': 'b', 'NODE_0000483': 'c'}}

In [14]:
# H3 has and H1 outgroup, and vice versa
outgroup = {'H3': 'CY179395.1', 'H1': 'CY180420.1'}

In [1]:
def get_tips_per_lineage(virus):
    """
    Given the common ancestor of the lienage, find all tips that are in the lineage
    """

    tree_file = f'data/full_ha_type_trees/{virus}.json'

    #read in the tree
    with open(tree_file, 'r') as f:
        tree_json = json.load(f)

    #put tree in Bio.phylo format
    tree = json_to_tree(tree_json)
    
    ancestral_nodes = lineage_ancestral_node[virus]
    
    tips_per_lineage = {}
    
    for node in tree.find_clades():
        if node.name in ancestral_nodes:
            tips_in_lineage = [x.name for x in node.get_terminals()]
            tips_per_lineage[node.name] = tips_in_lineage
            
    return tips_per_lineage

In [2]:
def parse_seqs_metadata(virus):
    """
    edit the sequence and metadata files from the full HA tree to just contain data for this lineage
    also include the outgroup sequence
    """
    
    full_seq_fasta = f'data/{virus}_sequences.fasta'
    
    full_meta = f'data/{virus}_metadata.tsv'
    
    tips_per_lineage = get_tips_per_lineage(virus)
    
    
    for anc, tips in tips_per_lineage.items():
        tips_and_outgroup = [outgroup[virus]] + tips
        
        lineage_letter = lineage_letters[virus][anc]
        seqs_this_lineage = []
        
        # get subset of sequences
        with open(full_seq_fasta, 'r') as h:
            for record in SeqIO.parse(h, 'fasta'):
                if record.id in tips_and_outgroup:
                    seqs_this_lineage.append(record)
                    
        SeqIO.write(seqs_this_lineage, 
                    f'data/{virus}{lineage_letter}_sequences.fasta', 
                    'fasta')
        
        # get a subset of the metadata
        all_meta = pd.read_csv(full_meta, sep='\t', dtype={"segment": str})
        
        filtered_meta = all_meta[all_meta["strain"].isin(tips_and_outgroup)]
                    
        filtered_meta.to_csv(f'data/{virus}{lineage_letter}_metadata.tsv', sep="\t", index=False)
        

In [31]:
parse_seqs_metadata('H1')

In [32]:
parse_seqs_metadata('H3')